# Same-batch interference between neuron updates

Teacher 8; widths 8 and 16; both layers train. We enumerate all neuron pairs on
each source minibatch, checking both orders. No training runs are started here.

**Finding:** strong interference is less frequent per eligible pair at width 16
early in momentum training, and aggregate marginal harm is smaller. The number
of events per batch is essentially unchanged, while severity relative to an
affected update's own benefit does not clearly improve. Aggregate harm is small.

See [the full report](../docs/pair_interference.md).


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT=Path.cwd()
if not (ROOT/'pyproject.toml').exists():
    ROOT=ROOT.parent
OUT=ROOT/'runs/pier_study/pairs'
summary=pd.read_csv(OUT/'pair_summary.csv')
effects=pd.read_csv(OUT/'pair_effects.csv')
seeds=pd.read_csv(OUT/'pair_seeds.csv')


## Exact test

For the source batch B, let $e=f-y$ and let $d_r$ be the exact prediction change
from updating neuron r alone. Define

$$A_r=\mathbb E_B(e d_r+\tfrac12d_r^2),\qquad I_{rs}=\mathbb E_B(d_rd_s).$$

The strong event $r\to s$ is $A_r<0$, $A_s<0$, and $A_s+I_{rs}>0$.
Both updates help alone, but adding s after r increases the batch loss.
I is symmetric, but the two orders can have different outcomes.

Each state uses 128 source batches of size 64. All 28/120 unordered pairs are
evaluated in both orders: 56/240 directed cases per batch. Momentum includes
the saved pre-step history. Full and 1% steps hold that history fixed.


## Frequency and two severity summaries

Eligible pairs have both standalone changes negative. With
$H_{rs}=\mathbf1_{\rm eligible}[A_s+I_{rs}]_+$:

- F = event count / eligible ordered-pair count.
- S = sum marginal harm / sum standalone second-neuron benefit over ALL eligible cases.
- S_event = sum marginal harm / sum standalone benefit restricted to interfering cases.

We pool batches within each seed before forming ratios, then average seeds.
The conditional S_event is undefined for seeds without events; valid counts are shown.
These statistics have different meanings and should not all be called event severity.


In [ ]:
early=summary[(summary.comparison=='mse_0.2')&(summary.probe_set=='original128')&(summary.scale==1)].copy()
early['frequency_percent']=100*early.frequency
early['aggregate_severity_percent']=100*early.severity
display(early[['optimizer','width','n_seeds','frequency_percent','aggregate_severity_percent',
               'event_severity','event_severity_n_valid','eligible_fraction']].round(7))
display(effects[(effects.comparison=='mse_0.2')&(effects.probe_set=='original128')&
                (effects.scale==1)&effects.metric.isin(['frequency','severity','event_severity'])]
        [['optimizer','metric','n','mean','low','high']].round(8))
display(Image(filename=str(OUT/'pair_interference.png')))


## Absolute event counts and harm

Width 16 supplies many more possible pairs. Its lower event probability does not
mean fewer events per batch. The absolute pairwise harm sum is nevertheless
smaller. This sum spans alternative two-neuron combinations that can overlap;
it is not an additive decomposition of the full update's loss change.


In [ ]:
display(early[['optimizer','width','events_per_batch','harm_per_batch','harm_over_full_gain',
               'event_benefit_share','event_severity']].round(9))


## Fresh-batch check and step length

The fresh check has 32 new source batches, using previously recomputed gradients
from draw seed 20261018, and the same 40 early training seeds. Strong interference
mostly disappears for 1% steps. The tiny-step momentum width difference does not
clearly replicate on the fresh draw.


In [ ]:
check=summary[(summary.comparison=='mse_0.2')&summary.probe_set.isin(['original128','fresh32'])].copy()
check['frequency_percent']=100*check.frequency
check['aggregate_severity_percent']=100*check.severity
display(check[['optimizer','probe_set','scale','width','frequency_percent','aggregate_severity_percent']].round(9))
display(Image(filename=str(OUT/'pair_step_sensitivity.png')))


## Later matched loss and equal steps

The later matched-loss comparison has eight paired SGD seeds, seven momentum
seeds, and five SGD-lr-0.1 control seeds. Matching is approximate and conditional
on available saved states. The momentum advantage is uncertain here; SGD has
rarer pair interference at width 16, but the aggregate harm remains tiny.

Equal-step states compare different training errors, so they cannot isolate width
at a common learning stage.


In [ ]:
later=summary[(summary.comparison=='mse_0.01')&(summary.probe_set=='original128')&(summary.scale==1)].copy()
later['frequency_percent']=100*later.frequency
later['aggregate_severity_percent']=100*later.severity
display(later[['optimizer','width','n_seeds','train_mse_mean','frequency_percent','aggregate_severity_percent','event_severity']].round(8))
timed=summary[(summary.comparison_kind=='equal_steps')&(summary.probe_set=='original128')&(summary.scale==1)].copy()
timed['step']=timed.comparison.str[5:].astype(int)
display(timed.sort_values(['optimizer','step','width'])
        [['optimizer','step','width','train_mse_mean','frequency','severity']].round(8))


## Verification and limits

Confidence intervals resample paired training seeds, not individual pairs or
batches. They are exploratory pointwise intervals conditional on one teacher and
dataset. The strict criterion excludes harmless diminishing returns, individually
harmful updates, and interactions requiring more than two neurons.

The direct audit covers 2,960 pair forwards across ten states and includes real
positive interference events. Raw arrays and per-pair/per-batch tables permit
recomputation. Sign tolerance is relative to source loss; float32 parameter
rounding is checked separately.


In [ ]:
display(json.loads((OUT/'audit.json').read_text()))
tolerance=pd.read_csv(OUT/'tolerance_sensitivity.csv')
display(tolerance[(tolerance.comparison=='mse_0.2')&(tolerance.scale==1)]
        .groupby(['optimizer','width','relative_tolerance'])[['frequency','severity']].mean())
